## 1. Data preparing and loading 

**Machine learning is the game of two things :** <br>
1. Converting the data into numerical value <br>
2. Building the model to predict the patterns / algorithms in the data

In [2]:
import torch
from torch import nn
import matplotlib.pyplot as plt

In [3]:
# Create known parameters
weight = 0.7
bais = 0.3

" Capital alpha are for matrix or tensors "<br>" Small alpha are for vector "


In [4]:
X = torch.arange(0,1,0.02).unsqueeze(dim=1)
y = weight*X + bais

In [5]:
y[:10] , X[:10]

(tensor([[0.3000],
         [0.3140],
         [0.3280],
         [0.3420],
         [0.3560],
         [0.3700],
         [0.3840],
         [0.3980],
         [0.4120],
         [0.4260]]),
 tensor([[0.0000],
         [0.0200],
         [0.0400],
         [0.0600],
         [0.0800],
         [0.1000],
         [0.1200],
         [0.1400],
         [0.1600],
         [0.1800]]))

In [6]:
X.shape , y.shape

(torch.Size([50, 1]), torch.Size([50, 1]))

In [7]:
len(X) , len(y)

(50, 50)

## 2. Splitting of data

In [8]:
train_split = int(0.8 * len(X))
X_train , y_train = X[: train_split] , y[: train_split]
X_test , y_test = X[train_split:] , y[train_split:]

In [9]:
len(X_train) , len(y_train) , len(X_test) , len(y_test) , 

(40, 40, 10, 10)

## 3. Visualizing the data

In [10]:
# def plot_predictions(train_data = X_train,
#                      train_label= y_train,
#                      test_data = X_test,
#                      test_label = y_test,
#                      predictions = None ):
# plt.scatter(X_train , y_train , colorizer='red', s=4)
# plt.scatter(X_test, y_test,colorizer='green' , s=4)
# plt.scatter(test_data , predictions , c='yellow' , s=4)

## 4. Building the model

" Create linear regression model "

In [11]:
class linear_regression_model(nn.Module):   # Almost everything in pytorch inheritence form nn.module
    def __init__(self):
        super().__init__()                  # Calls the parent class (nn.Module) initializer. Required for PyTorch to properly track parameters and model behavior.
        self.weights = nn.Parameter(torch.randn(1,
                                               requires_grad=True,
                                               dtype=torch.float))
        self.bais = nn.Parameter(torch.randn(1,
                                            dtype=torch.float))
        
    def forward(self , x: torch.Tensor) -> torch.tensor:   # it tells us that model will input tensor and output tensor 
        return self.weights * x + self.bais
        

**Pytorch model building essentials :** <br>
1. torch.nn - contains all of the buildings for computational graphs. ( neural network == CG)<br>
2. torch.nn.Parameter - what parameters should our model try and learn.<br>
3. torch.nn.module - The base class for all neural network modules, if you subclass it, you should overwrite forward()<br>
4. torch.optim - this where the optimizer in Pytorch live, they help with gradient descent<br>
5. def forward() - All nn.module subclasses require you to overwrite forward(), this method defines what happens in the forward computation

## 5. Checking the contents of our Pytorch model

In [12]:
torch.manual_seed(42)          # if we remove this we will get the diff values each time

model_0 = linear_regression_model()
list(model_0.parameters())

[Parameter containing:
 tensor([0.3367], requires_grad=True),
 Parameter containing:
 tensor([0.1288], requires_grad=True)]

In [13]:
model_0.state_dict()

OrderedDict([('weights', tensor([0.3367])), ('bais', tensor([0.1288]))])

## 6. Making predictions using 'torch.inference.mode()'

" inference.model()" will track only the necessery data and make the model more faster. While, the gradient descent will keep the record of all the data

In [14]:
with torch.inference_mode():
    y_pred = model_0(X_test)

y_pred

tensor([[0.3982],
        [0.4049],
        [0.4116],
        [0.4184],
        [0.4251],
        [0.4318],
        [0.4386],
        [0.4453],
        [0.4520],
        [0.4588]])

In [15]:
with torch.no_grad():
    y_pred = model_0(X_test)

y_pred

tensor([[0.3982],
        [0.4049],
        [0.4116],
        [0.4184],
        [0.4251],
        [0.4318],
        [0.4386],
        [0.4453],
        [0.4520],
        [0.4588]])

## 7. Training the model

**1. Some useful functions :**

- Loss function : It is used to measure , how wrong your model's predictions are to the ideal one.<br>
- Optimiser: Takes into account the loss of the model and adjust the parameters accordingly. <br>

**2. And specifically for pytorch , we need:**

- A training loop<br>
- A testing loop

**3. Inside an optimizer :**

- params - The model parameters you'd like to optimize.
- lr (learning rate) - the learning rate is a hyperparameter used to define how big/small the optimizer changes the parameters with each step

In [16]:
# Setup a loss function
loss_fn = nn.L1Loss()

# Setup an optimizer
optimiser = torch.optim.SGD(params=model_0.parameters(),
                               lr=0.01)

In [17]:
model_0.state_dict()

OrderedDict([('weights', tensor([0.3367])), ('bais', tensor([0.1288]))])

## 8. Building a training loop

Gradient descent - The ratio of y/X , meaning , The change of y over the change in X

Hyperparameters - parameters that we set by ourselves

EPOCH - it is the no of loops through the data. it pass the data through the model for a number of epochs

In [18]:
torch.manual_seed(42)
epochs = 168
epoch_count = []
train_loss = []
test_loss = []

#0 - Loop through the data
for epoch in range(epochs):
    print(f"Epoch : {epoch}----------")
    model_0.train()

    #1 - forward passs
    y_pred = model_0(X_train)

    #2 - Calculate the loss
    loss = loss_fn(y_pred , y_train)

    #3 - Optimizer zero grad
    optimiser.zero_grad()

    #4 - Perform backpropagation on the loss with respect to the parameters of the model
    loss.backward()

    #5 - Step the optimizer ( perform gradient descent )
    optimiser.step()

    #6 - Testing
    model_0.eval()             # It turn off the gradient tracking and some settings that aren't needed or evaltuion  , torch.inference does the same ( with extra)
    with torch.inference_mode():
        y_pred_new =  model_0(X_test)
        t_loss = loss_fn(y_pred_new , y_test)
        print(f'loss : {loss} | Test loss : {t_loss}')
        epoch_count.append(epoch)
        train_loss.append(loss.detach().cpu().numpy())
        test_loss.append(t_loss.detach().cpu().numpy())

model_0.state_dict()
    

Epoch : 0----------
loss : 0.31288138031959534 | Test loss : 0.48106518387794495
Epoch : 1----------
loss : 0.3013603389263153 | Test loss : 0.4675942063331604
Epoch : 2----------
loss : 0.28983935713768005 | Test loss : 0.4541231691837311
Epoch : 3----------
loss : 0.2783183455467224 | Test loss : 0.44065219163894653
Epoch : 4----------
loss : 0.26679736375808716 | Test loss : 0.4271811842918396
Epoch : 5----------
loss : 0.2552763521671295 | Test loss : 0.41371020674705505
Epoch : 6----------
loss : 0.24375534057617188 | Test loss : 0.40023916959762573
Epoch : 7----------
loss : 0.23223432898521423 | Test loss : 0.3867681920528412
Epoch : 8----------
loss : 0.22071333229541779 | Test loss : 0.37329721450805664
Epoch : 9----------
loss : 0.20919232070446014 | Test loss : 0.3598262071609497
Epoch : 10----------
loss : 0.1976713240146637 | Test loss : 0.3463551998138428
Epoch : 11----------
loss : 0.18615034222602844 | Test loss : 0.3328842222690582
Epoch : 12----------
loss : 0.1746293

OrderedDict([('weights', tensor([0.6947])), ('bais', tensor([0.3028]))])

In [19]:
weight , bais

(0.7, 0.3)

In [20]:
# plt.plot(epoch_count , train_loss, c='red')
# plt.plot(epoch_count , test_loss , c='yellow')
# plt.xlabel('no of epochs')
# plt.ylabel('Loss rate')

## 9. Saving a model

There are three main methods to save and load a model :

1. `torch.save()` - allows you save a pytorch object in python's pickle format
2. `torch.load()` - allows you to load a saved pytorch object
3. `torch.nn.Module.load_state_dict()`  - this allows to load a model's saved state dictionary


In [21]:
from pathlib import Path

model_path = Path('models')
model_path.mkdir(parents=True , exist_ok=True)

model_name = 'linear_regression_model_0.pth'
model_save_path = model_path/model_name

model_save_path

torch.save(obj=model_0.state_dict(),
           f=model_save_path)

## 10. Loading a model

In [25]:
loaded_model_0 = linear_regression_model()
loaded_model_0.load_state_dict(torch.load(f=model_save_path))

<All keys matched successfully>

In [26]:
loaded_model_0.state_dict()

OrderedDict([('weights', tensor([0.6947])), ('bais', tensor([0.3028]))])

In [30]:
## Make some predicitons with our loaded model :
loaded_model_0.eval()
with torch.inference_mode():
    loaded_model_pred  = loaded_model_0(X_test)

loaded_model_pred == y_pred_new

tensor([[True],
        [True],
        [True],
        [True],
        [True],
        [True],
        [True],
        [True],
        [True],
        [True]])